Load data

In [ ]:
import sqlite3
import pandas as pd

#sql database path
db_path = "challenge.db"

#connect to database
conn = sqlite3.connect(db_path)

#load tables into pandas dataframes
five_hundred_cities_df = pd.read_sql("SELECT * FROM five_hundred_cities;", conn)
access_df = pd.read_sql("SELECT * FROM access;", conn)
variable_list_df = pd.read_sql("SELECT * FROM variable_list;", conn)

Join access and five_hundred_cities table 

In [ ]:
#Prepare data for joining

# Convert FIPS and TractFIPS to string 
access_df['FIPS'] = access_df['FIPS'].astype(str)
five_hundred_cities_df['TractFIPS'] = five_hundred_cities_df['TractFIPS'].astype(str)

#Add leading 0s to FIPS in Access table
access_df['FIPS'] = access_df['FIPS'].str.zfill(5)

#Add leading 0s to TractFIPS in 500 Cities table
five_hundred_cities_df['TractFIPS'] = five_hundred_cities_df['TractFIPS'].str.zfill(11)

#Take first 5 characters of TractFIPS in 500 Cities table
five_hundred_cities_df['CountyFIPS'] = five_hundred_cities_df['TractFIPS'].str[:5]

In [ ]:
#Join both tables on FIPS and CountyFIPS
joined_df = pd.merge(access_df, five_hundred_cities_df, how='inner', left_on='FIPS', right_on='CountyFIPS')
joined_df.head()

In [ ]:
#summary statistics
joined_df.describe(include='all')

Variable selection on joined data

In [ ]:
#Pull all variables and variable codes from variable_list table
variable_names = variable_list_df['Variable Name'].tolist()
variable_codes = variable_list_df['Variable Code'].tolist()
for name, code in zip(variable_names, variable_codes):
    print(f"{name}: {code}")

In [ ]:
#determine which variables are present in joined table
for col in joined_df.columns:
    print(col)

In [ ]:
#Select most important variables required for further analysis (selected manually) and create a new dataframe with only those variables
selected_variables = ["State", "County", "PlaceName", "Place_TractID", "Population2010", "LACCESS_POP15", "LACCESS_LOWI15", "PCT_LACCESS_LOWI15", "LACCESS_HHNV15", "PCT_LACCESS_HHNV15", "LACCESS_SNAP15", "PCT_LACCESS_SNAP15", "LACCESS_CHILD15", "PCT_LACCESS_CHILD15", "LACCESS_SENIORS15", "PCT_LACCESS_SENIORS15", "GROC16", "SUPERCLUB_2016", "CONVENIENCE_2016", "SNAP_AUTH_2017", "WIC_AUTH_2016", "FASTFOOD_2016", "FULLSERVICE_2016", "FARMERS_MARKETS_2018", "SNAP_FARMERS_2018", "Geolocation"]

selected_variables = [
    "State",
    "County",
    "PlaceName",
    "Place_TractID",
    "Population2010",
    "LACCESS_POP10", 
    "LACCESS_POP15", 
    "PCT_LACCESS_POP10", 
    "PCT_LACCESS_POP15", 
    "LACCESS_LOWI15", 
    "LACCESS_HHNV15", 
    "LACCESS_SNAP15", 
    "PCT_LACCESS_SNAP15", 
    "LACCESS_CHILD15", 
    "PCT_LACCESS_CHILD15", 
    "LACCESS_SENIORS15", 
    "PCT_LACCESS_SENIORS15", 
    "ACCESS2_CrudePrev", 
    "OBESITY_CrudePrev", 
    "DIABETES_CrudePrev", 
    "CANCER_CrudePrev", 
    "BINGE_CrudePrev", 
    "CSMOKING_CrudePrev", 
    "COPD_CrudePrev", 
    "MHLTH_CrudePrev"
]
#create the new df
joined2_df = joined_df[selected_variables]

print(joined2_df.head())


Mark variables based on type of metric they are (county or tractFIPS)

In [ ]:
#Rename columns for additional clarity
#Add F_ to county based variables (FIPS)
county_rename = [
    "LACCESS_POP10", "LACCESS_POP15", "PCT_LACCESS_POP10", "PCT_LACCESS_POP15",
    "LACCESS_LOWI15", "LACCESS_HHNV15", "LACCESS_SNAP15", "PCT_LACCESS_SNAP15",
    "LACCESS_CHILD15", "PCT_LACCESS_CHILD15", "LACCESS_SENIORS15", "PCT_LACCESS_SENIORS15"
]
renamed_county = {col: f"F_{col}" for col in county_rename}
joined2_df.rename(columns=renamed_county, inplace=True)

#Add TF_ to tract based variables (TractFIPS)
tract_rename = [
    "Population2010",
    "ACCESS2_CrudePrev", 
    "OBESITY_CrudePrev", 
    "DIABETES_CrudePrev", 
    "CANCER_CrudePrev", 
    "BINGE_CrudePrev", 
    "CSMOKING_CrudePrev", 
    "COPD_CrudePrev", 
    "MHLTH_CrudePrev"
]
renamed_tract = {col: f"TF_{col}" for col in tract_rename}
joined2_df.rename(columns=renamed_tract, inplace=True)

Creating table and scoring mechanism to identify most necessary locations for program

In [ ]:
#Drop rows with missing values
#These appear to be for variables that are in 2015 and there are 38 of them. We want to work with the most recent data.

joined2_df = joined2_df.dropna()

In [ ]:
#Create a variable that measures the change in population from 2015 to 2010
#This will give us a very rough idea of if the problem is worsening over time.
#A positive value means the population is growing, a negative value means the population is shrinking 

joined2_df["F_PopChange"] = (joined2_df["F_LACCESS_POP15"] - joined2_df["F_LACCESS_POP10"])

In [ ]:
#Create a scoring mechanism to identify the area that needs the program the most
#Will not include percentage variables as actual amount variables are available and our goal is to help as many people as possible
#We will not include the 2010 population either as it will be redundant with the 2015 population and the change in population


In [ ]:
#First we need to adjust the percent value variables to become actual population values
#This can be done by multiplying it by Population2010 / 100 (as it needs to be in decimal form)

joined3_df = joined2_df.copy()
joined3_df["TF_ACCESS2_CrudePrev_Pop"] = (joined2_df["TF_ACCESS2_CrudePrev"] / 100 * joined2_df["TF_Population2010"])
joined3_df["TF_OBESITY_CrudePrev_Pop"] = (joined2_df["TF_OBESITY_CrudePrev"] / 100 * joined2_df["TF_Population2010"])
joined3_df["TF_DIABETES_CrudePrev_Pop"] = (joined2_df["TF_DIABETES_CrudePrev"] / 100 * joined2_df["TF_Population2010"])
joined3_df["TF_CANCER_CrudePrev_Pop"] = (joined2_df["TF_CANCER_CrudePrev"] / 100 * joined2_df["TF_Population2010"])
joined3_df["TF_BINGE_CrudePrev_Pop"] = (joined2_df["TF_BINGE_CrudePrev"] / 100 * joined2_df["TF_Population2010"])
joined3_df["TF_CSMOKING_CrudePrev_Pop"] = (joined2_df["TF_CSMOKING_CrudePrev"] / 100 * joined2_df["TF_Population2010"])
joined3_df["TF_COPD_CrudePrev_Pop"] = (joined2_df["TF_COPD_CrudePrev"] / 100 * joined2_df["TF_Population2010"])
joined3_df["TF_MHLTH_CrudePrev_Pop"] = (joined2_df["TF_MHLTH_CrudePrev"] / 100 * joined2_df["TF_Population2010"])

#drop the old variables
joined3_df = joined3_df.drop(columns=[
    "TF_ACCESS2_CrudePrev", 
    "TF_OBESITY_CrudePrev", 
    "TF_DIABETES_CrudePrev", 
    "TF_CANCER_CrudePrev", 
    "TF_BINGE_CrudePrev", 
    "TF_CSMOKING_CrudePrev", 
    "TF_COPD_CrudePrev", 
    "TF_MHLTH_CrudePrev"
])

In [ ]:

#Create weighting for each variable
#We have access variables, health variables, and population change
#Access to services should likely be higher priority than health outcomes as this can be due to various other factors such as lifestyle choices
#Although population change is important, it should be less important than access to services as these are the immediate needs of the people
weighting = {
    "F_PopChange": .1,
    "F_LACCESS_POP15": .12,
    "F_LACCESS_LOWI15": .12,
    "F_LACCESS_HHNV15": .12,
    "F_LACCESS_SNAP15": .12,
    "F_LACCESS_CHILD15": .12,
    "F_LACCESS_SENIORS15": .12,
    "TF_ACCESS2_CrudePrev_Pop": .05,
    "TF_OBESITY_CrudePrev_Pop": .05,
    "TF_DIABETES_CrudePrev_Pop": .05,
    "TF_CANCER_CrudePrev_Pop": .05,
    "TF_BINGE_CrudePrev_Pop": .05,
    "TF_CSMOKING_CrudePrev_Pop": .05,
    "TF_COPD_CrudePrev_Pop": .05,
    "TF_MHLTH_CrudePrev_Pop": .05
}

#Create the score variable
scored_df = joined3_df.copy()
scored_df["Score"] = joined3_df[list(weighting.keys())].apply(lambda row: sum(row * pd.Series(weighting)), axis=1)


Examine locations for program by county

In [ ]:
#We will identify the counties with the highest scores to launch the program in
#We will need to sum certain variables to get the totals for each county and rescore

#Start by dropping tract variables that cannot be summed.
scored_county_df = scored_df.copy()
scored_county_df = scored_county_df.drop(columns=[
    "PlaceName",
    "Place_TractID",
    "Score"])

In [ ]:
#Columns that start with TF can now be summed to get the totals for each county
#We will update the scoring mechanism after

#Select columns that start with TF
columns_tf = [col for col in scored_county_df.columns if col.startswith('TF_')]

#select columns to keep and group by
columns_keep = [col for col in scored_county_df.columns if not col.startswith('TF_')]

#Group by county and sum the columns
scored_county_df = scored_county_df.groupby(columns_keep, as_index=False)[columns_tf].sum()


In [ ]:
#Perform scoring for scored_county_df
scored_county_df["Score"] = scored_county_df[list(weighting.keys())].apply(lambda row: sum(row * pd.Series(weighting)), axis=1)

In [ ]:
#Add back FIPS column
scored_county_df = pd.merge(scored_county_df, access_df[['State', 'County', 'FIPS']], on=['State', 'County'], how='left')

In [ ]:
#Visualizations
#Chloropleth map of the areas with the highest scores
#Demographic data for the areas with the highest scores


Visualizations

Chloropleth Map to Highlight Areas Best for the Program

In [ ]:

#Plotly code
import plotly.express as px
import json
from urllib.request import urlopen

with urlopen('https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json') as response:
    counties = json.load(response)

fig = px.choropleth(scored_county_df,
                    geojson=counties,
                    locations='FIPS', 
                    color='Score', 
                    scope = "usa",
                    hover_name='County',  
                    hover_data=["State", "Score"],  
                    color_continuous_scale="YlOrRd",  
                    title="Choropleth Map of Scored Counties"
)

fig.update_geos(fitbounds="locations") 
fig.show()

Bar graphs highlight which counties have the highest scores

In [ ]:
import matplotlib.pyplot as plt

# Select the top 5 counties by score and create labels for the plot
top_5_counties = scored_county_df.nlargest(5, 'Score')

# Create labels that combine County, State, and FIPS
labels = top_5_counties['County'] + ', ' + top_5_counties['State'] + ' (' + top_5_counties['FIPS'].astype(str) + ')'

#Darker colors to emphasize top 3
colors = ['orange' if i < 3 else 'lightblue' for i in range(len(top_5_counties))]

plt.figure(figsize=(15, 6))

# Create a bar plot with set colors
plt.bar(range(len(top_5_counties)), 
        top_5_counties['Score'], 
        color=colors)

# Add county names for x-axis labels
plt.xticks(range(len(top_5_counties)), 
           labels,
           rotation=45,
           ha='right')

# Add the score on top of each bar
for i, score in enumerate(top_5_counties['Score']):
    plt.text(i, score, f"{score:.2f}", ha='center', va='bottom', fontweight='bold')

# Add labels and title with bold font
plt.xlabel('Counties', fontweight='bold', fontsize=14)
plt.ylabel('Score', fontweight='bold', fontsize=14)
plt.title('Top 5 Counties by Score', fontweight='bold', fontsize=16)

plt.tight_layout()
plt.show()



Plot Displaying People in Each of the Top 3 Counties

In [ ]:
import matplotlib.pyplot as plt

# Select the top 3 counties by score
top_3_counties = scored_county_df.nlargest(3, 'Score')

# Sort the top 3 counties by population
top_3_counties_sorted = top_3_counties.sort_values('F_LACCESS_POP15', ascending=False)

# Create labels
labels = top_3_counties_sorted['County'] + ', ' + top_3_counties_sorted['State'] + ' (' + top_3_counties_sorted['FIPS'].astype(str) + ')'

# Create the bar plot
plt.figure(figsize=(10, 6))
bars = plt.bar(range(len(top_3_counties_sorted)), 
               top_3_counties_sorted['F_LACCESS_POP15'], color='skyblue')

# Add the 2015 population with low access to stores on top of each bar
for i, F_LACCESS_POP15 in enumerate(top_3_counties_sorted['F_LACCESS_POP15']):
    plt.text(i, F_LACCESS_POP15, f"{F_LACCESS_POP15:.2f}", ha='center', va='bottom', fontweight='bold')

# Add labels and title with bold font
plt.xticks(range(len(top_3_counties_sorted)), 
           labels,
           rotation=45,
           ha='right')
plt.xlabel('Counties', fontweight='bold', fontsize=14)
plt.ylabel('Population, Low Access to Stores, 2015', fontweight='bold', fontsize=14)
plt.title('Top 3 Counties by Population, Low Access to Stores, 2015', fontweight='bold', fontsize=16)

plt.tight_layout()
plt.show()
